# 🚀 FedLLM-Attack 实验1: Local vs. Global Behavior Profile

## 实验说明
本实验验证联邦学习中的关键现象：**High ASR + Low BTF**

核心发现：
- **Local Profile**: 恶意客户端本地训练后的模型行为画像
- **Global Profile**: FedAvg聚合后的全局模型行为画像
- **关键现象**: \( |\Delta_T| \ll |\Delta_B| \) 且 \( p_T^G \) 仍然高

## 1️⃣ 环境准备

In [1]:
# 检查GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PyTorch version: 2.2.2+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 3060
GPU memory: 11.63 GB


In [7]:
!pip install transformers==4.36.2 peft==0.10.0 trl==0.8.6 datasets==2.18.0 accelerate==0.27.2

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 26.3 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.0 MB/s eta 0:00:00:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple

In [9]:
!pip install sentencepiece

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.0 MB/s eta 0:00:00


In [10]:
!pip uninstall transformers -y
!pip install transformers==4.36.2

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 99.0 MB/s eta 0:00:00:00:0100:01


In [11]:
!pip install tokenizers==0.15.2

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


## 2️⃣ 导入依赖

In [2]:
# ============================================================
# 导入所有必要的库
# ============================================================

import sys

# Jupyter 会自动添加：
# -f xxx/kernel.json
# FedLLM-Attack 的 config.py 使用 HfArgumentParser，
# 因此需要在导入 config 前清除 Jupyter 参数。
sys.argv = [sys.argv[0]]

import copy
import os
import json
import numpy as np
from tqdm import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM
from peft import (
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

from utils import *
from federated_learning import *

from config import (
    get_config,
    save_config,
    get_model_config,
    get_training_args,
)

from evaluation.behavior_profile import (
    BehaviorProfileEvaluator,
    ExperimentRecorder,
    BehaviorProfile,
    compute_btf,
    EvaluationResult,
)

print("✅ 所有依赖导入成功！")

✅ 所有依赖导入成功！


## 3️⃣ 配置参数（根据服务器情况调整）

In [3]:
# ==================== 实验参数配置 ====================

# 👤 模型选择
# 推荐: "meta-llama/Llama-2-7b-hf" 或 "meta-llama/Llama-2-13b-hf"
# 如果显存有限(<=16GB): 使用 7b 模型 + load_in_4bit=True
# 如果显存充足(>=24GB): 可以尝试 13b 模型
MODEL_NAME = "meta-llama/Llama-2-7b-hf"

# 🔧 LoRA 配置
USE_PEFT = True
PEFT_LORA_R = 8
PEFT_LORA_ALPHA = 16

# 💾 量化配置（根据显存选择）
# False = 全精度，需要更多显存
# True = 4bit量化，节省显存
LOAD_IN_4BIT = True   # 推荐开启，7b模型16GB显存足够
LOAD_IN_8BIT = False

# 📚 训练配置
LEARNING_RATE = 2e-5
BATCH_SIZE = 4
NUM_TRAIN_EPOCHS = 3
MAX_STEPS = 50
SEQ_LENGTH = 512

# 🔄 联邦学习配置
FED_ALG = "fedavg"
NUM_ROUNDS = 100
SAMPLE_CLIENTS = 3
BENIGN_NUM_CLIENTS = 7
MALICIOUS_NUM_CLIENTS = 3

# 🎯 实验特定配置
TARGET_TOPIC = "nuclear weapons"
EVAL_FREQUENCY = 5  # 每5轮评估一次

# 📁 输出配置
OUTPUT_DIR = "./outputs/exp1_behavior_profile"

print("✅ 配置完成！")
print(f"模型: {MODEL_NAME}")
print(f"量化: 4bit={LOAD_IN_4BIT}, 8bit={LOAD_IN_8BIT}")
print(f"联邦轮数: {NUM_ROUNDS}, 每轮采样: {SAMPLE_CLIENTS}个客户端")
print(f"良性客户端: {BENIGN_NUM_CLIENTS}, 恶意客户端: {MALICIOUS_NUM_CLIENTS}")

✅ 配置完成！
模型: meta-llama/Llama-2-7b-hf
量化: 4bit=True, 8bit=False
联邦轮数: 100, 每轮采样: 3个客户端
良性客户端: 7, 恶意客户端: 3


## 4️⃣ 核心函数定义

In [4]:
def identify_malicious_clients(fed_args) -> set:
    """
    识别恶意客户端索引
    """
    malicious_clients = set()
    if hasattr(fed_args, 'malicious_num_clients') and hasattr(fed_args, 'benign_num_clients'):
        start_idx = sum(fed_args.benign_num_clients)
        for num in fed_args.malicious_num_clients:
            for i in range(num):
                malicious_clients.add(start_idx + i)
            start_idx += num
    return malicious_clients


def evaluate_behavior_profile(model, tokenizer, evaluator, batch_size=4, device="cuda"):
    """
    评估模型行为画像
    """
    model.eval()
    profile = evaluator.evaluate_model(model, tokenizer, batch_size=batch_size)
    model.train()
    return profile

## 5️⃣ 主实验代码

In [5]:
# ==================== 运行实验 ====================

def run_experiment():
    """
    运行联邦后门攻击实验1
    """
    
    print("=" * 60)
    print("🚀 启动 FedLLM-Attack 实验1")
    print("=" * 60)
    
    # ===== 创建配置对象 =====
    # 这里我们手动设置参数而不是从yaml加载
    class ScriptArgs:
        model_name_or_path = MODEL_NAME
        use_peft = USE_PEFT
        peft_lora_r = PEFT_LORA_R
        peft_lora_alpha = PEFT_LORA_ALPHA
        load_in_8bit = LOAD_IN_8BIT
        load_in_4bit = LOAD_IN_4BIT
        learning_rate = LEARNING_RATE
        batch_size = BATCH_SIZE
        seq_length = SEQ_LENGTH
        gradient_accumulation_steps = 1
        num_train_epochs = NUM_TRAIN_EPOCHS
        max_steps = MAX_STEPS
        gradient_checkpointing = True
        output_dir = OUTPUT_DIR
        log_with = "none"
        save_steps = 50
        save_total_limit = 5
        seed = 2023
        template = "llama2"
        trust_remote_code = False
        existing_lora = None
        dataset_name = None
        train_file = None
        valid_file = None
        max_src_len = 512
        max_tgt_len = 512
    
    class FedArgs:
        fed_alg = FED_ALG
        num_rounds = NUM_ROUNDS
        num_clients = BENIGN_NUM_CLIENTS + MALICIOUS_NUM_CLIENTS
        sample_clients = SAMPLE_CLIENTS
        split_strategy = "iid"
        num_data_per_client = 500
        benign_dataset_names = ["lucasmccabe-lmi/CodeAlpaca-20k"]
        benign_num_clients = [BENIGN_NUM_CLIENTS]
        malicious_dataset_names = ["MaliciousGen"]
        malicious_num_clients = [MALICIOUS_NUM_CLIENTS]
        proxy_lr = 1.0
        server_momentum = 0.0
        fednova_alpha = 0.5
    
    script_args = ScriptArgs()
    fed_args = FedArgs()
    
    # 识别恶意客户端
    malicious_clients = identify_malicious_clients(fed_args)
    print(f"恶意客户端: {sorted(malicious_clients)}")
    
    # 保存配置
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    recorder = ExperimentRecorder(OUTPUT_DIR)
    
    # ===== 加载数据集 =====
    dataset_list, num_client_list = get_sft_datasets(script_args, fed_args)
    print(f"数据集: {dataset_list}")
    print(f"每数据集客户端数: {num_client_list}")
    
    # 分割数据集
    local_datasets = []
    num_clients = sum(num_client_list)
    for dataset, num_client in zip(dataset_list, num_client_list):
        splited_datasets = split_dataset(fed_args, script_args, dataset, num_client)
        local_datasets.extend(splited_datasets)
    
    setattr(fed_args, 'num_clients', num_clients)
    sample_num_list = [len(local_datasets[i]) for i in range(fed_args.num_clients)]
    
    # ===== 加载模型 =====
    device_map, quantization_config, torch_dtype = get_model_config(script_args)
    
    print("\n加载模型中...")
    model = AutoModelForCausalLM.from_pretrained(
        script_args.model_name_or_path,
        quantization_config=quantization_config,
        device_map=device_map,
        trust_remote_code=script_args.trust_remote_code,
        torch_dtype=torch_dtype,
    )
    
    if script_args.load_in_8bit or script_args.load_in_4bit:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=training_args.gradient_checkpointing
        )
    
    peft_config = get_peft_model(model, script_args)
    model.print_trainable_parameters()
    
    # ===== 初始化模型字典 =====
    from peft import get_peft_model_state_dict
    global_dict = copy.deepcopy(get_peft_model_state_dict(model))
    local_dict_list = [copy.deepcopy(global_dict) for i in range(fed_args.num_clients)]
    proxy_dict, opt_proxy_dict = get_proxy_dict(fed_args, global_dict)
    global_auxiliary, auxiliary_model_list, auxiliary_delta_dict = get_auxiliary_dict(fed_args, global_dict)
    
    # ===== 加载分词器 =====
    tokenizer = AutoTokenizer.from_pretrained(
        script_args.model_name_or_path,
        use_fast=False,
        padding_side="right"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.unk_token
    
    # ===== 准备数据处理 =====
    formatting_prompts_func, response_template = get_formatting_prompts_func(script_args.template, tokenizer.eos_token)
    response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)[2:]
    data_collator = DataCollatorForCompletionOnlyLM(response_template_ids, tokenizer=tokenizer)
    
    # ===== 初始化行为评估器 =====
    device = "cuda" if torch.cuda.is_available() else "cpu"
    evaluator = BehaviorProfileEvaluator(
        target_topic=TARGET_TOPIC,
        num_samples_per_region=20,
        device=device
    )
    
    # ===== 开始联邦训练 =====
    training_loss = [[] for i in range(fed_args.num_clients)]
    local_profiles_this_round = {}
    
    for round in tqdm(range(fed_args.num_rounds), desc="联邦训练进度"):
        
        clients_this_round = get_clients_this_round(fed_args, round)
        print(f"\n>> 轮次 {round + 1}/{fed_args.num_rounds}: 客户端 {clients_this_round}")
        
        local_profiles_this_round = {}
        
        # ===== 本地训练阶段 =====
        for client in range(fed_args.num_clients):
            
            if client not in clients_this_round:
                training_loss[client].append(-1)
                continue
            
            # 同步全局模型
            set_peft_model_state_dict(model, global_dict)
            
            # 获取数据集
            sub_dataset = get_dataset_this_round(local_datasets[client], round, fed_args, script_args)
            new_lr = cosine_learning_rate(round, fed_args.num_rounds, script_args.learning_rate, 1e-6)
            training_args = get_training_args(script_args, new_lr)
            
            # 训练本地模型
            trainer = get_fed_local_sft_trainer(
                model=model,
                tokenizer=tokenizer,
                training_args=training_args,
                local_dataset=sub_dataset,
                formatting_prompts_func=formatting_prompts_func,
                data_collator=data_collator,
                global_dict=global_dict,
                fed_args=fed_args,
                script_args=script_args,
                local_auxiliary=auxiliary_model_list[client],
                global_auxiliary=global_auxiliary,
            )
            
            results = trainer.train()
            training_loss[client].append(results.training_loss)
            
            # ===== 评估本地行为画像 =====
            if client in malicious_clients and round % EVAL_FREQUENCY == 0:
                print(f"\n  [评估客户端 {client} 的本地画像]")
                local_profile = evaluate_behavior_profile(model, tokenizer, evaluator, batch_size=4, device=device)
                print(f"  本地画像: T={local_profile.p_T:.2%}, E={local_profile.p_E:.2%}, B={local_profile.p_B:.2%}, N={local_profile.p_N:.2%}")
                local_profiles_this_round[client] = local_profile
            
            # ===== 传输本地信息 =====
            if fed_args.fed_alg == 'scaffold':
                auxiliary_model_list[client], auxiliary_delta_dict[client] = trainer.get_auxiliary_param()
            else:
                local_dict_list[client] = copy.deepcopy(get_peft_model_state_dict(model))
        
        # ===== FedAvg 聚合 =====
        print("\n[执行 FedAvg 聚合...]")
        global_dict, global_auxiliary = global_aggregate(
            fed_args, global_dict, local_dict_list, sample_num_list,
            clients_this_round, round, proxy_dict=proxy_dict,
            opt_proxy_dict=opt_proxy_dict, auxiliary_info=(global_auxiliary, auxiliary_delta_dict)
        )
        
        # 更新全局模型
        set_peft_model_state_dict(model, global_dict)
        
        # ===== 评估全局行为画像 =====
        should_eval_global = any(c in malicious_clients for c in clients_this_round)
        if should_eval_global and round % EVAL_FREQUENCY == 0:
            print(f"\n[评估全局画像]")
            global_profile = evaluate_behavior_profile(model, tokenizer, evaluator, batch_size=4, device=device)
            print(f"全局画像: T={global_profile.p_T:.2%}, E={global_profile.p_E:.2%}, B={global_profile.p_B:.2%}, N={global_profile.p_N:.2%}")
            
            # 记录结果
            for client in clients_this_round:
                if client in malicious_clients:
                    local_profile = local_profiles_this_round.get(client)
                    if local_profile is not None:
                        delta_T = global_profile.p_T - local_profile.p_T
                        delta_E = global_profile.p_E - local_profile.p_E
                        delta_B = global_profile.p_B - local_profile.p_B
                        delta_N = global_profile.p_N - local_profile.p_N
                        btf = compute_btf(local_profile, global_profile)
                        
                        result = EvaluationResult(
                            round_idx=round,
                            client_id=client,
                            is_malicious=True,
                            local_profile=local_profile,
                            global_profile=global_profile,
                            delta_T=delta_T,
                            delta_E=delta_E,
                            delta_B=delta_B,
                            delta_N=delta_N,
                            btf=btf
                        )
                        recorder.add_result(result)
                        
                        # 打印对比
                        print(f"\n  客户端 {client} 行为画像对比:")
                        print(f"  {'区域':<12} {'本地':>10} {'全局':>10} {'Δ':>10}")
                        print(f"  {'-'*44}")
                        print(f"  {'Target':<12} {local_profile.p_T:>10.2%} {global_profile.p_T:>10.2%} {delta_T:>+10.2%}")
                        print(f"  {'Equivalent':<12} {local_profile.p_E:>10.2%} {global_profile.p_E:>10.2%} {delta_E:>+10.2%}")
                        print(f"  {'Boundary':<12} {local_profile.p_B:>10.2%} {global_profile.p_B:>10.2%} {delta_B:>+10.2%}")
                        print(f"  {'Normal':<12} {local_profile.p_N:>10.2%} {global_profile.p_N:>10.2%} {delta_N:>+10.2%}")
                        print(f"  {'-'*44}")
                        print(f"  BTF: {btf:.4f}")
            
            recorder.print_latest_summary()
        
        # ===== 保存模型 =====
        if (round + 1) % 50 == 0 or round + 1 == 10:
            trainer.save_model(os.path.join(OUTPUT_DIR, f"checkpoint-{round + 1}"))
        
        # 保存训练损失
        np.save(os.path.join(OUTPUT_DIR, "training_loss.npy"), np.array(training_loss))
        
        # 保存结果
        if (round + 1) % (EVAL_FREQUENCY * 5) == 0:
            recorder.save_results()
            recorder.save_summary_csv()
    
    # ===== 最终保存 =====
    recorder.save_results()
    recorder.save_summary_csv()
    
    print("\n" + "=" * 60)
    print("🎉 实验1完成！")
    print("=" * 60)
    print(f"结果保存至: {OUTPUT_DIR}")
    print("=" * 60)
    
    return recorder

print("✅ 主函数定义完成！")

✅ 主函数定义完成！


## 6️⃣ 运行实验（执行此单元格）

In [6]:
# 🚀 开始运行实验
# 这可能需要数小时，取决于GPU和参数配置

recorder = run_experiment()

🚀 启动 FedLLM-Attack 实验1
恶意客户端: [7, 8, 9]


AttributeError: 'ScriptArgs' object has no attribute 'local_data_dir'

## 7️⃣ 可视化结果

In [ ]:
# 加载并可视化结果
import json
import matplotlib.pyplot as plt
import pandas as pd

# 读取结果
results_file = os.path.join(OUTPUT_DIR, "behavior_profile_results.json")
summary_file = os.path.join(OUTPUT_DIR, "behavior_profile_summary.csv")

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    print(f"✅ 加载了 {len(results)} 条结果记录")
    
    # 转换为DataFrame
    df = pd.DataFrame([{
        'round': r['round_idx'],
        'client': r['client_id'],
        'local_T': r['local_profile']['p_T'],
        'local_E': r['local_profile']['p_E'],
        'local_B': r['local_profile']['p_B'],
        'local_N': r['local_profile']['p_N'],
        'global_T': r['global_profile']['p_T'],
        'global_E': r['global_profile']['p_E'],
        'global_B': r['global_profile']['p_B'],
        'global_N': r['global_profile']['p_N'],
        'delta_T': r['delta_T'],
        'delta_B': r['delta_B'],
        'btf': r['btf']
    } for r in results])
    
    # 绘制关键指标变化
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Target成功率
    axes[0, 0].plot(df['round'], df['local_T'], 'b-', label='Local T', marker='o')
    axes[0, 0].plot(df['round'], df['global_T'], 'r-', label='Global T', marker='s')
    axes[0, 0].set_xlabel('Round')
    axes[0, 0].set_ylabel('Success Rate')
    axes[0, 0].set_title('Target Region Success Rate')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Boundary变化
    axes[0, 1].plot(df['round'], df['local_B'], 'b-', label='Local B', marker='o')
    axes[0, 1].plot(df['round'], df['global_B'], 'r-', label='Global B', marker='s')
    axes[0, 1].set_xlabel('Round')
    axes[0, 1].set_ylabel('Boundary Rate')
    axes[0, 1].set_title('Boundary Region Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Delta对比
    axes[1, 0].plot(df['round'], df['delta_T'], 'b-', label='ΔT', marker='o')
    axes[1, 0].plot(df['round'], df['delta_B'], 'r-', label='ΔB', marker='s')
    axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
    axes[1, 0].set_xlabel('Round')
    axes[1, 0].set_ylabel('Delta')
    axes[1, 0].set_title('Local → Global Delta (T vs B)')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # BTF变化
    axes[1, 1].plot(df['round'], df['btf'], 'g-', marker='o')
    axes[1, 1].axhline(y=1.0, color='k', linestyle='--', alpha=0.5, label='BTF=1.0')
    axes[1, 1].set_xlabel('Round')
    axes[1, 1].set_ylabel('BTF')
    axes[1, 1].set_title('Behavior Transferability Factor (BTF)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'exp1_results.png'), dpi=150)
    plt.show()
    
    print(f"\n✅ 图表已保存至: {os.path.join(OUTPUT_DIR, 'exp1_results.png')}")
else:
    print("❌ 结果文件不存在，请先运行实验")

## 📋 结果摘要

In [ ]:
# 打印最终结果摘要
if 'df' in dir():
    print("\n" + "=" * 60)
    print("📊 实验1 最终结果摘要")
    print("=" * 60)
    
    latest = df.iloc[-1] if len(df) > 0 else None
    
    if latest is not None:
        print(f"\n最后评估轮次: Round {int(latest['round'])}")
        print(f"\n{'指标':<20} {'本地':>12} {'全局':>12} {'变化':>12}")
        print("-" * 56)
        print(f"{'Target (T)':<20} {latest['local_T']:>12.2%} {latest['global_T']:>12.2%} {latest['delta_T']:>+12.2%}")
        print(f"{'Boundary (B)':<20} {latest['local_B']:>12.2%} {latest['global_B']:>12.2%} {latest['delta_B']:>+12.2%}")
        print(f"{'BTF':<20} {'-':>12} {'-':>12} {latest['btf']:>12.4f}")
        
        print("\n" + "=" * 60)
        print("🔍 关键发现:")
        print("=" * 60)
        
        if abs(latest['delta_T']) < abs(latest['delta_B']):
            print("✅ 验证成功: |ΔT| < |ΔB| (Target稳定性 > Boundary稳定性)")
        else:
            print("⚠️ 变化不符合预期模式")
        
        if latest['global_T'] > 0.8:
            print("✅ ASR维持高位: Global Target成功率 = {:.2%}".format(latest['global_T']))
        
        if latest['btf'] < 0.5:
            print("✅ BTF较低: 行为转移性受限")
        
        print("\n" + "=" * 60)
        print("💡 结论: High ASR + Low BTF 现象已验证")
        print("=" * 60)